In [1]:
# Install Dependencies
!pip install -q streamlit prophet plotly pyngrok
!npm install localtunnel


changed 1 package in 4s

3 packages are looking for funding
  run `npm fund` for details


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from prophet import Prophet
from datetime import datetime, timedelta

# Page Config
st.set_page_config(page_title="FitPulse Health Insights", layout="wide")

st.title("FitPulse: Health Anomaly Detection ")


# Sidebar
st.sidebar.header("Data Configuration")
uploaded_file = st.sidebar.file_uploader("Upload Fitness Data (CSV)", type=['csv'])

# Synthetic Data Generator (Fallback)
def generate_synthetic_data():
    dates = pd.date_range(end=datetime.now(), periods=60, freq='D')
    ids = ['P001'] * 60

    # Base metrics with noise
    steps = np.random.normal(8000, 2000, 60)
    steps = [max(0, s) for s in steps]

    hr = np.random.normal(70, 5, 60)

    sleep = np.random.normal(420, 60, 60) # Minutes (7 hours)

    # Inject Anomalies
    # HR Spikes
    hr[10] += 30
    hr[45] += 25

    # Sleep Deprivation
    sleep[20] -= 200
    sleep[21] -= 180

    # Low Activity
    steps[30] = 500

    df = pd.DataFrame({
        'Id': ids,
        'Date': dates,
        'step_count': steps,
        'heart_rate': hr,
        'sleep_tracking': sleep
    })
    df['timestamp'] = df['Date'] # Match milestone 3 format roughly
    return df

if uploaded_file is not None:
    try:
        df = pd.read_csv(uploaded_file)
        # Ensure timestamp handling
        if 'timestamp' not in df.columns:
             # Try to find a date column
             date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
             if date_cols:
                 df['timestamp'] = pd.to_datetime(df[date_cols[0]])
             else:
                 st.error("CSV must have a 'timestamp' or 'Date' column.")
                 st.stop()
        else:
            df['timestamp'] = pd.to_datetime(df['timestamp'])

        st.sidebar.success("File Uploaded Successfully")
    except Exception as e:
        st.error(f"Error reading file: {e}")
        st.stop()
else:
    st.info("Using Demo Data (Upload a CSV to analyze your own files)")
    df = generate_synthetic_data()

# Data Preview
with st.expander("View Raw Data"):
    st.dataframe(df.head())

# Analysis Section
st.header("Anomaly Detection Analysis")

metric_map = {
    "Heart Rate": "heart_rate",
    "Sleep Duration": "sleep_tracking",
    "Step Count": "step_count"
}

selected_metric_label = st.selectbox("Select Metric to Analyze", list(metric_map.keys()))
selected_metric = metric_map[selected_metric_label]

if selected_metric not in df.columns:
    st.error(f"Column '{selected_metric}' not found in dataset. Available columns: {list(df.columns)}")
else:
    # Filtering
    min_date = df['timestamp'].min().date()
    max_date = df['timestamp'].max().date()

    start_date, end_date = st.sidebar.date_input("Date Range", [min_date, max_date])

    mask = (df['timestamp'].dt.date >= start_date) & (df['timestamp'].dt.date <= end_date)
    filtered_df = df.loc[mask].copy()

    if st.button("Detect Anomalies"):
        with st.spinner("Running Prophet Model... (This may take a moment)"):
            # Prepare for Prophet
            prophet_df = filtered_df[['timestamp', selected_metric]].rename(columns={'timestamp': 'ds', selected_metric: 'y'})
            prophet_df = prophet_df.sort_values('ds')

            # Model
            # Using strict thresholds to catch anomalies easily for demo
            model = Prophet(daily_seasonality=True, yearly_seasonality=False, weekly_seasonality=True)
            model.fit(prophet_df)

            future = model.make_future_dataframe(periods=0) # Checking historical data for anomalies
            forecast = model.predict(future)

            # Merge and Calculate Residuals
            results = prophet_df.merge(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds')
            results['residual'] = results['y'] - results['yhat']

            # Dynamic Threshold (2 Standard Deviations as per Milestone 3)
            std_dev = results['residual'].std()
            threshold = 2 * std_dev

            results['anomaly'] = abs(results['residual']) > threshold
            results['anomaly_label'] = results['anomaly'].map({True: 'Anomalous', False: 'Normal'})

            anomalies = results[results['anomaly'] == True]

            # Visualization
            st.subheader(f"{selected_metric_label} Trends & Anomalies")

            fig = go.Figure()

            # Actual Data
            fig.add_trace(go.Scatter(x=results['ds'], y=results['y'], mode='lines', name='Actual', line=dict(color='blue', width=1)))

            # Forecast
            fig.add_trace(go.Scatter(x=results['ds'], y=results['yhat'], mode='lines', name='Expected (Trend)', line=dict(color='green', width=1, dash='dash')))

            # Anomalies
            fig.add_trace(go.Scatter(
                x=anomalies['ds'],
                y=anomalies['y'],
                mode='markers',
                name='Anomaly',
                marker=dict(color='red', size=8, symbol='x')
            ))

            fig.update_layout(title=f"{selected_metric_label} over Time", hovermode="x unified")
            st.plotly_chart(fig, use_container_width=True)

            # Metrics
            c1, c2, c3 = st.columns(3)
            c1.metric("Total Data Points", len(results))
            c2.metric("Anomalies Detected", len(anomalies))
            c3.metric("Anomaly %", f"{len(anomalies)/len(results)*100:.2f}%")

            # Report Generation
            st.subheader("Anomaly Report")
            if not anomalies.empty:
                st.dataframe(anomalies[['ds', 'y', 'yhat', 'residual']])

                # Download
                csv = anomalies.to_csv(index=False).encode('utf-8')
                st.download_button(
                    "Download Anomaly Report (CSV)",
                    csv,
                    "anomaly_report.csv",
                    "text/csv",
                    key='download-csv'
                )
            else:
                st.write("No anomalies detected in the selected range.")

Overwriting app.py


In [ ]:
# Run the Streamlit App
print("Starting Streamlit...")

import subprocess
import time
from pyngrok import ngrok

# Start Streamlit in the background
# The port must be explicitly set for Streamlit and matched by ngrok
# Added --server.enableCORS and --server.enableXsrfProtection for broader compatibility.
streamlit_command = ["streamlit", "run", "app.py", "--server.port", "8501", "--server.enableCORS", "true", "--server.enableXsrfProtection", "false"]
streamlit_process = subprocess.Popen(streamlit_command)

# Give Streamlit a few seconds to start up
time.sleep(5)


ngrok.set_auth_token("36FLiazE0JrIWKnEos96vhL46Uy_7fp8RRiZHLqzv6NMC9Cfy")
# -------------------

try:
    # Open a ngrok tunnel to the Streamlit port
    public_url = ngrok.connect(8501, bind_tls=True) # Use bind_tls=True for HTTPS tunnel
    print(f"Streamlit App is running at: {public_url}")
    print("To stop the Streamlit app and ngrok tunnel, interrupt this cell execution.")
    print("If you encounter issues, ensure your ngrok auth token is set (see comments in code).")

    
    while True:
        time.sleep(60) # Sleep for a minute to keep the process alive
except Exception as e:
    print(f"An error occurred with ngrok: {e}")
    if "authentication token" in str(e).lower():
        print("ngrok requires an authentication token. Please get one from https://dashboard.ngrok.com/get-started/your-authtoken")
        print("Then set it using: `from pyngrok import ngrok; ngrok.set_auth_token('YOUR_NGROK_AUTH_TOKEN')`.")
    elif "command not found" in str(e).lower() or "ngrok binary not found" in str(e).lower():
        print("The ngrok executable might not be installed or in your PATH. pyngrok usually handles this, but if not, try `ngrok.install_ngrok()`.")
    # Terminate streamlit process if ngrok fails
    streamlit_process.terminate()
    streamlit_process.wait()

Starting Streamlit...
